In [21]:
import os
import json
from pathlib import Path

def original_dataset_filter(s: str):
    not_xslx = not s.endswith(".xlsx")
    not_dotfile = not s.startswith(".")
    return not_xslx and not_dotfile

def patient_dir_filter(s: str):
    not_datafile = s != "DATAFILE"
    not_dotfile = not s.startswith(".")
    return not_datafile and not_dotfile

def jpg_filter(s: str):
    return s.lower().endswith(".jpg")

def list_all_jpgs(dir_path: str):
    jpgs_list = []

    for root, dirs, files in os.walk(dir_path):
        for file in files:
            orig_path = Path(os.path.join(root, file))
            without_prefix_path = Path(*orig_path.parts[2:])
            jpgs_list.append(str(without_prefix_path))
    
    jpgs_list = list(filter(jpg_filter, jpgs_list))

    return jpgs_list

base_path = "nuclear-cataract-original"
original_patients_dirs = os.listdir(base_path)
original_patients_dirs = list(filter(original_dataset_filter, original_patients_dirs))

patients = {}

# IZQ - left side
# DER - right side

print(original_patients_dirs)
for dir in original_patients_dirs:
    patient_dir_path = os.path.join(base_path, dir)
    eyes_dirs = os.listdir(patient_dir_path)
    eyes_dirs_lower = list(map(lambda s: s.lower(), eyes_dirs))
    left_eyes_paths = []
    right_eyes_paths = []

    try:
        izq_dir_name = eyes_dirs[eyes_dirs_lower.index("izq")]
        izq_dir_path = os.path.join(patient_dir_path, izq_dir_name)
        left_eyes_paths.extend(list_all_jpgs(izq_dir_path))
    except ValueError:
        pass

    try:
        der_dir_name = eyes_dirs[eyes_dirs_lower.index("der")]
        der_dir_path = os.path.join(patient_dir_path, der_dir_name)
        right_eyes_paths.extend(list_all_jpgs(der_dir_path))
    except ValueError:
        pass

    patients[dir] = {
        "left": left_eyes_paths,
        "right": right_eyes_paths,
    }

patients_images_json = json.dumps(patients, indent=4)
with open("patients.json", "w+") as f:
    f.write(patients_images_json)

['61', '95', '59', '92', '66', '104', '50', '68', '103', '57', '03', '04', '32', '35', '102', '69', '56', '105', '51', '58', '67', '93', '94', '60', '34', '33', '05', '02', '20', '18', '27', '11', '29', '16', '42', '89', '45', '111', '73', '87', '80', '74', '28', '17', '10', '19', '26', '21', '75', '81', '86', '72', '44', '110', '43', '88', '07', '38', '36', '09', '31', '91', '65', '62', '96', '109', '100', '54', '107', '98', '53', '30', '37', '08', '01', '06', '39', '106', '99', '52', '101', '55', '97', '108', '63', '64', '90', '46', '79', '112', '41', '115', '83', '77', '48', '70', '84', '24', '23', '15', '12', '85', '71', '76', '82', '49', '40', '114', '47', '113', '78', '13', '14', '22', '25']


In [22]:
from pprint import pprint
import matplotlib.pyplot as plt
import numpy as np
import tqdm
import time

our_basepath = "nuclear-cataract-ours"
our_images_dir = os.path.join(our_basepath, "images")

original_images_paths = []
our_images_paths = []
our_images_bytes = {}

original_to_our = {}
our_to_original = {}

for patient_id in patients.keys():
    for image_path in (patients[patient_id]["left"] + patients[patient_id]["right"]):
        original_image_path = os.path.join(base_path, patient_id, image_path)
        original_images_paths.append(original_image_path)

for our_image_name in os.listdir(our_images_dir):
    our_image_path = os.path.join(our_images_dir, our_image_name)
    our_images_paths.append(our_image_path)

for our_image_path in tqdm.tqdm(our_images_paths, desc="loading our images"):
    with open(our_image_path, "rb") as f:
        our_image_bytes = f.read()
    our_images_bytes[our_image_path] = our_image_bytes

loading our images: 100%|██████████| 1339/1339 [00:00<00:00, 2247.53it/s]


In [23]:
def rempthpref(pth: str, n: int) -> str:
   pth_parts = pth.split("/")
   pth_parts = list(filter(lambda s: s != "", pth_parts))
   return "/".join(pth_parts[n:])

for original_image_path in tqdm.tqdm(original_images_paths, desc="comparing images"):
    with open(original_image_path, "rb") as f:
        original_image_bytes = f.read()

    for i, our_image_path in enumerate(our_images_paths):
        our_image_bytes = our_images_bytes[our_image_path]
        #are_identical = np.array_equal(original_image_mat, our_image_mat)
        are_identical = original_image_bytes == our_image_bytes

        if are_identical:
            original_to_our[rempthpref(original_image_path, 1)] = rempthpref(our_image_path, 2)
            our_to_original[rempthpref(our_image_path, 2)] = rempthpref(original_image_path, 1)
            our_images_paths.remove(our_image_path)
            break

with open("our_to_original.json", "w+") as f:
    f.write(json.dumps(our_to_original, indent=4))

with open("original_to_our.json", "w+") as f:
    f.write(json.dumps(original_to_our, indent=4))

comparing images: 100%|██████████| 1320/1320 [00:00<00:00, 1940.70it/s]


In [24]:
import copy

patients_ours = copy.deepcopy(patients)

for patient_id in patients:
    patients_ours[patient_id]["left"] = list(map(lambda s: original_to_our[os.path.join(patient_id, s)], patients[patient_id]["left"]))
    patients_ours[patient_id]["right"] = list(map(lambda s: original_to_our[os.path.join(patient_id, s)], patients[patient_id]["right"]))

with open("patients_ours.json", "w+") as f:
    f.write(json.dumps(patients_ours, indent=4))